# 37 — Operating Regimes

**Engineering statement:** Operating regimes specify adaptive confidence scheduling.

Notebook 29 introduced hardware feasibility constraints. Notebook 37 synthesizes the repository so far: confidence profiles, throughput objectives, and hardware constraints define operating regimes. Each regime selects a different serving policy.


## Repository roadmap

```text
00 Context
07 Verification Resource
13 Confidence Scheduling
17 Semi-Autoregressive Design
23 Throughput Objective
29 Hardware-Aware Scheduling
37 Operating Regimes
43 Resource Allocation
49 Adaptive AI Infrastructure
```


## Start here

```text
confidence profiles
→ throughput objective
→ hardware constraints
→ feasible schedules
→ operating regimes
→ adaptive serving policy
```

Notebook 37 asks:

> **Given current confidence, load, memory pressure, and latency constraints, which operating regime is the serving system in?**

This turns confidence-scheduled decoding into a regime-selection problem rather than a single fixed scheduling rule.


In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle
from IPython.display import FileLink, display

CWD = Path.cwd().resolve()
if (CWD / "figures").exists() or (CWD / "results").exists() or (CWD / "notebooks").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "figures").exists() or (CWD.parent / "results").exists() or (CWD.parent / "notebooks").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "37_operating_regimes"
FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

def savefig(name):
    path = FIGURES / name
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    return path

def rounded_box(ax, xy, w, h, text, fontsize=12):
    box = FancyBboxPatch(
        xy, w, h,
        boxstyle="round,pad=0.03,rounding_size=0.08",
        linewidth=1.4,
        edgecolor="black",
        facecolor="white"
    )
    ax.add_patch(box)
    ax.text(xy[0]+w/2, xy[1]+h/2, text, ha="center", va="center", fontsize=fontsize)
    return box

def arrow(ax, start, end):
    ax.annotate("", xy=end, xytext=start, arrowprops=dict(arrowstyle="->", lw=1.8))


## 1. Operating-regime map

The previous notebooks defined variables; Notebook 37 organizes them into regimes.

A regime is a region of serving-state space where the same scheduling logic remains appropriate.

Examples:

- **confidence-limited:** draft prefixes decay quickly;
- **compute-limited:** verification capacity is scarce;
- **memory-limited:** KV/cache traffic restricts longer schedules;
- **latency-limited:** service-level targets force shorter schedules;
- **throughput-balanced:** confidence, compute, and hardware constraints are well matched.


In [ ]:
# 37_operating_regime_map.png

fig, ax = plt.subplots(figsize=(8.2, 7))
ax.axis("off")
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)

# axes
ax.annotate("", xy=(3.5, 0), xytext=(-3.5, 0), arrowprops=dict(arrowstyle="<->", lw=2))
ax.annotate("", xy=(0, 3.5), xytext=(0, -3.5), arrowprops=dict(arrowstyle="<->", lw=2))

ax.text(0, 3.65, "Memory bound", ha="center", va="bottom", fontsize=15, fontweight="bold")
ax.text(0, -3.65, "Latency bound", ha="center", va="top", fontsize=15, fontweight="bold")
ax.text(-3.65, 0, "Compute\nbound", ha="right", va="center", fontsize=15, fontweight="bold")
ax.text(3.65, 0, "Confidence\nbound", ha="left", va="center", fontsize=15, fontweight="bold")

rounded_box(ax, (-1.15, -0.45), 2.3, 0.9, "Throughput-\nbalanced", fontsize=13)
rounded_box(ax, (-3.2, 2.1), 1.7, 0.75, "memory +\ncompute", fontsize=11)
rounded_box(ax, (1.5, 2.1), 1.7, 0.75, "memory +\nconfidence", fontsize=11)
rounded_box(ax, (-3.2, -2.9), 1.7, 0.75, "latency +\ncompute", fontsize=11)
rounded_box(ax, (1.5, -2.9), 1.7, 0.75, "latency +\nconfidence", fontsize=11)

ax.set_title("Operating regimes specify adaptive confidence scheduling", fontsize=20, pad=20)
savefig("37_operating_regime_map.png")


## 2. Phase diagram

A practical controller needs a map from serving state to policy.

A useful first state space is:

- \(x\): draft predictability / confidence floor;
- \(y\): active serving load.

Low predictability pushes the system into a confidence-limited regime. High load pushes it into memory and latency-limited regimes.


In [ ]:
# 37_phase_diagram.png

load = np.linspace(4, 64, 80)
predictability = np.linspace(0.2, 0.9, 90)
P, L = np.meshgrid(predictability, load)

# Regime labels:
# 0 confidence-limited, 1 throughput-balanced, 2 compute-limited, 3 memory-limited, 4 latency-limited
regime = np.zeros_like(P, dtype=int)
regime[(P > 0.42) & (L < 24)] = 1
regime[(P > 0.55) & (L >= 24) & (L < 42)] = 2
regime[(P > 0.52) & (L >= 42) & (P < 0.72)] = 3
regime[(L >= 42) & (P >= 0.72)] = 4
regime[(P < 0.38) & (L > 36)] = 4

fig, ax = plt.subplots(figsize=(9.5, 6.5))
im = ax.imshow(regime, aspect="auto", origin="lower",
               extent=[predictability.min(), predictability.max(), load.min(), load.max()])
ax.set_xlabel("draft predictability / confidence floor", fontsize=14)
ax.set_ylabel("active requests R", fontsize=14)
ax.set_title("Operating-regime phase diagram", fontsize=20)
cbar = plt.colorbar(im, ax=ax, ticks=[0,1,2,3,4])
cbar.ax.set_yticklabels(["confidence", "balanced", "compute", "memory", "latency"])
cbar.set_label("selected regime")

# Labels in regions
ax.text(0.27, 18, "confidence-\nlimited", fontsize=12, ha="center")
ax.text(0.62, 15, "balanced", fontsize=12, ha="center")
ax.text(0.64, 32, "compute-\nlimited", fontsize=12, ha="center")
ax.text(0.59, 52, "memory-\nlimited", fontsize=12, ha="center")
ax.text(0.80, 52, "latency-\nlimited", fontsize=12, ha="center")

savefig("37_phase_diagram.png")

phase_data = {
    "load": load.round(3).tolist(),
    "predictability": predictability.round(3).tolist(),
    "regime_labels": ["confidence-limited", "throughput-balanced", "compute-limited", "memory-limited", "latency-limited"]
}
(RESULTS / "37_phase_diagram_metadata.json").write_text(json.dumps(phase_data, indent=2), encoding="utf-8")


## 3. Constraint activation

As serving load rises, different constraints become active. The important point is not that one global policy wins. The scheduler should switch policies as constraints activate.

A deployment can therefore be represented by a constraint vector:

\[
z=(C_{\rm conf},C_{\rm compute},C_{\rm memory},C_{\rm latency}).
\]


In [ ]:
# 37_constraint_activation.png

loads = np.array([8, 20, 36, 52])
labels = ["low load", "moderate", "high", "saturated"]

confidence = np.array([0.65, 0.72, 0.78, 0.82])
compute = np.array([0.35, 0.55, 0.78, 0.95])
memory = np.array([0.25, 0.42, 0.72, 0.92])
latency = np.array([0.20, 0.38, 0.70, 0.98])

x = np.arange(len(labels))
width = 0.19

fig, ax = plt.subplots(figsize=(10.5, 5.8))
ax.bar(x - 1.5*width, confidence, width, label="confidence")
ax.bar(x - 0.5*width, compute, width, label="compute")
ax.bar(x + 0.5*width, memory, width, label="memory")
ax.bar(x + 1.5*width, latency, width, label="latency")
ax.axhline(0.75, linestyle="--", linewidth=1.5, label="active threshold")
ax.set_xticks(x, labels)
ax.set_ylim(0, 1.05)
ax.set_ylabel("constraint pressure", fontsize=14)
ax.set_title("Constraint activation changes the scheduling regime", fontsize=20)
ax.legend(ncol=3)
ax.grid(axis="y", alpha=0.3)
savefig("37_constraint_activation.png")


## 4. Policy switching

Operating regimes matter because they select controllers. A fixed confidence threshold is not enough.

A controller should:

1. estimate the current regime;
2. select the appropriate scheduling policy;
3. apply a feasible verification schedule;
4. update as traffic and confidence distributions shift.


In [ ]:
# 37_policy_switching.png

fig, ax = plt.subplots(figsize=(12.5, 4.8))
ax.axis("off")
ax.set_xlim(0, 12.5)
ax.set_ylim(0, 4)

nodes = [
    ("request\nstream", 0.4, 2.2, 1.35, 0.75),
    ("state\nestimator", 2.35, 2.2, 1.45, 0.75),
    ("regime\nclassifier", 4.45, 2.2, 1.55, 0.75),
    ("policy\nselector", 6.65, 2.2, 1.4, 0.75),
    ("verification\nschedule", 8.75, 2.2, 1.65, 0.75),
    ("serving\noutcome", 10.95, 2.2, 1.25, 0.75),
]
for label, x0, y0, w, h in nodes:
    rounded_box(ax, (x0, y0), w, h, label, fontsize=12)
for i in range(len(nodes)-1):
    _, x0, y0, w, h = nodes[i]
    _, x1, y1, w1, h1 = nodes[i+1]
    arrow(ax, (x0+w+0.05, y0+h/2), (x1-0.05, y1+h1/2))

signals = [
    ("confidence", 2.2, 0.65, 1.25, 0.55),
    ("load", 3.7, 0.65, 0.9, 0.55),
    ("memory", 4.85, 0.65, 1.0, 0.55),
    ("latency", 6.1, 0.65, 1.0, 0.55),
]
for label, x0, y0, w, h in signals:
    rounded_box(ax, (x0, y0), w, h, label, fontsize=10)
    arrow(ax, (x0+w/2, y0+h+0.04), (5.2, 2.15))

# feedback
ax.annotate("", xy=(2.95, 3.1), xytext=(11.6, 3.1), arrowprops=dict(arrowstyle="->", lw=1.6))
ax.text(7.2, 3.28, "online feedback", ha="center", fontsize=11)

ax.set_title("Adaptive confidence scheduling selects policies by operating regime", fontsize=20, pad=12)
savefig("37_policy_switching.png")


## 5. Throughput landscape

Notebook 23 optimized \(\Theta(\ell)\). Notebook 37 treats the optimum as a ridge in a changing landscape.

The useful engineering question is:

> As load and predictability change, where does the optimal schedule move?


In [ ]:
# 37_throughput_landscape.png

ell = np.arange(1, 10)
loads = np.array([8, 20, 36, 52])
predictability = 0.62

def prefix_survival(q, ell):
    cs = np.clip(q + 0.16*np.exp(-0.18*np.arange(ell)), 0.05, 0.98)
    return np.cumprod(cs).sum()

def latency_proxy(R, ell):
    return 1.0 + 0.018*(R**1.12) + 0.06*ell + 0.0028*R*ell

def throughput_proxy(R, ell, q):
    return prefix_survival(q, ell) / latency_proxy(R, ell)

fig, ax = plt.subplots(figsize=(10.5, 6.2))
for R0 in loads:
    vals = np.array([throughput_proxy(R0, e, predictability) for e in ell])
    ax.plot(ell, vals, marker="o", label=f"R={R0}")
    opt_i = int(np.argmax(vals))
    ax.scatter([ell[opt_i]], [vals[opt_i]], s=90, zorder=4)
    ax.text(ell[opt_i]+0.08, vals[opt_i], f"$\\ell^*={ell[opt_i]}$", fontsize=10)
ax.set_xlabel("scheduled prefix length $\\ell$", fontsize=14)
ax.set_ylabel("throughput proxy", fontsize=14)
ax.set_title("Throughput landscape: optimum shifts with serving load", fontsize=20)
ax.grid(True, alpha=0.3)
ax.legend(title="active requests")
savefig("37_throughput_landscape.png")


## 6. Regime transition graph

As load rises or confidence falls, the system moves through regimes. The graph is not a strict sequence in every deployment, but it provides a useful engineering map.

The controller's job is to detect the active regime and apply the right scheduling behavior.


In [ ]:
# 37_regime_transition_graph.png

fig, ax = plt.subplots(figsize=(12, 4.4))
ax.axis("off")
ax.set_xlim(0, 12)
ax.set_ylim(0, 4)

nodes = [
    ("Throughput-\nbalanced", 0.6, 2.1, 1.7, 0.75),
    ("Confidence-\nlimited", 3.0, 2.1, 1.7, 0.75),
    ("Compute-\nlimited", 5.4, 2.1, 1.7, 0.75),
    ("Memory-\nlimited", 7.8, 2.1, 1.7, 0.75),
    ("Latency-\nlimited", 10.0, 2.1, 1.7, 0.75),
]
for label, x0, y0, w, h in nodes:
    rounded_box(ax, (x0, y0), w, h, label, fontsize=11)
for i in range(len(nodes)-1):
    _, x0, y0, w, h = nodes[i]
    _, x1, y1, w1, h1 = nodes[i+1]
    arrow(ax, (x0+w+0.05, y0+h/2), (x1-0.05, y1+h1/2))

# backward/adaptive arrows
ax.annotate("", xy=(3.85, 1.92), xytext=(6.25, 1.92), arrowprops=dict(arrowstyle="<->", lw=1.2))
ax.annotate("", xy=(6.25, 1.65), xytext=(8.65, 1.65), arrowprops=dict(arrowstyle="<->", lw=1.2))
ax.text(6.25, 1.1, "policy can move backward when load drops or confidence improves", ha="center", fontsize=10)

ax.set_title("Regime transition graph for adaptive serving", fontsize=20, pad=12)
savefig("37_regime_transition_graph.png")


## 7. Repository summary

The first seven notebooks form a complete engineering path:

```text
context
→ resource
→ confidence
→ draft architecture
→ throughput
→ hardware constraints
→ operating regimes
```

This is the point where local mechanisms become a deployment-scale map.


In [ ]:
# 37_repository_summary.png

fig, ax = plt.subplots(figsize=(13, 4.6))
ax.axis("off")
ax.set_xlim(0, 13)
ax.set_ylim(0, 4)

nodes = [
    ("00\nContext", 0.4, 2.2, 1.25, 0.75),
    ("07\nVerification\nResource", 2.1, 2.2, 1.35, 0.75),
    ("13\nConfidence\nScheduling", 3.95, 2.2, 1.45, 0.75),
    ("17\nSemi-AR\nDrafting", 5.95, 2.2, 1.35, 0.75),
    ("23\nThroughput\nObjective", 7.8, 2.2, 1.45, 0.75),
    ("29\nHardware\nConstraints", 9.8, 2.2, 1.35, 0.75),
    ("37\nOperating\nRegimes", 11.55, 2.2, 1.25, 0.75),
]
for label, x0, y0, w, h in nodes:
    rounded_box(ax, (x0, y0), w, h, label, fontsize=10)
for i in range(len(nodes)-1):
    _, x0, y0, w, h = nodes[i]
    _, x1, y1, w1, h1 = nodes[i+1]
    arrow(ax, (x0+w+0.05, y0+h/2), (x1-0.05, y1+h1/2))

rounded_box(ax, (3.6, 0.65), 6.1, 0.75, "Local mechanisms become deployment-scale operating regimes.", fontsize=13)

ax.set_title("Confidence-scheduled decoding: repository synthesis through Notebook 37", fontsize=20, pad=12)
savefig("37_repository_summary.png")


## Key equations

Prefix survival:

\[
a_j=\prod_{k=1}^{j}c_k
\]

Throughput objective:

\[
\Theta(\ell)=A(\ell)S(B)
\]

Feasible schedule set:

\[
\mathcal{F}
=
\{\ell:\;L(R,\ell)\le L_{\max},\;M(R,\ell)\le M_{\max}\}
\]

Regime classification:

\[
\rho
=
g(C_{\rm conf},C_{\rm compute},C_{\rm memory},C_{\rm latency})
\]

Regime-conditioned policy:

\[
\pi_\rho
:
(s_{\rm confidence},s_{\rm hardware})
\mapsto
\ell^*
\]

Adaptive schedule:

\[
\ell^*
=
\pi_{\rho}(s_t).
\]


## Engineering summary

Notebook 37 completes the first systems arc.

A confidence-scheduled serving system should not use one fixed rule. It should identify the active operating regime and select a policy consistent with confidence, compute, memory, and latency constraints.

> **Operating regimes specify adaptive confidence scheduling.**


In [ ]:
# Save notebook manifest

manifest = {
    "notebook": "37_operating_regimes.ipynb",
    "title": "Operating Regimes",
    "engineering_statement": "Operating regimes specify adaptive confidence scheduling.",
    "figures": [
        "37_operating_regime_map.png",
        "37_phase_diagram.png",
        "37_constraint_activation.png",
        "37_policy_switching.png",
        "37_throughput_landscape.png",
        "37_regime_transition_graph.png",
        "37_repository_summary.png",
    ],
    "next_notebook": "43_resource_allocation.ipynb",
}
(RESULTS / "37_operating_regimes_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
manifest


## Download artifacts

Run the final cell to package Notebook 37 outputs for download.

The archive includes:

- `37_operating_regimes.ipynb`
- generated figures
- generated JSON outputs
- notebook manifest


In [ ]:
# Package Notebook 37 artifacts for download

from pathlib import Path
from IPython.display import FileLink, display
import zipfile

CWD = Path.cwd().resolve()
if (CWD / "figures").exists() or (CWD / "results").exists() or (CWD / "notebooks").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "figures").exists() or (CWD.parent / "results").exists() or (CWD.parent / "notebooks").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

NOTEBOOKS = REPO_ROOT / "notebooks"
FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "37_operating_regimes"
SITE = REPO_ROOT / "site"

RESULTS.mkdir(parents=True, exist_ok=True)

zip_path = RESULTS / "37_operating_regimes_artifacts.zip"

notebook_candidates = [
    NOTEBOOKS / "37_operating_regimes.ipynb",
    Path.cwd() / "37_operating_regimes.ipynb",
]
notebook_path = next((p for p in notebook_candidates if p.exists()), None)

paths_to_include = []
if notebook_path is not None:
    paths_to_include.append(notebook_path)

for item in [FIGURES, RESULTS, SITE]:
    if item.exists():
        paths_to_include.append(item)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_include:
        item = Path(item)
        if item.is_dir():
            for path in item.rglob("*"):
                if not path.is_file():
                    continue
                if path.resolve() == zip_path.resolve():
                    continue
                try:
                    archive_name = path.relative_to(REPO_ROOT)
                except ValueError:
                    archive_name = path.name
                zf.write(path, archive_name)
        elif item.exists():
            try:
                archive_name = item.relative_to(REPO_ROOT)
            except ValueError:
                archive_name = item.name
            zf.write(item, archive_name)

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")
display(FileLink(str(zip_path)))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass
